# P12 — Synapse: relasjonell forståelse

Kjører C0, C1, C3 og C5 på 40 frosne oppgaver.

Legg disse i Colab Secrets:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GEMINI_API_KEY`
- `GITHUB_TOKEN` med lesetilgang til det private Tofoo-repoet

Base task pack SHA-256: `e5eec7868ba9e8737c3f6a9200036819b97b9659df0a4e7c628c8d95ccffc05a`

Effektiv task pack SHA-256 etter transparent fasitkorreksjon: `0fa9a53a144cdc89875cc864c88d90d6c2cb535cff01dd86bad21be85311a6d9`


In [ ]:
%pip -q install "openai>=1.0.0" "anthropic>=0.40.0" "google-genai>=1.0.0" "tenacity>=8.2.0" "pandas>=2.0.0" "matplotlib>=3.7.0"

In [ ]:
import base64
import hashlib
import importlib.util
import json
import os
import sys
from getpass import getpass
from pathlib import Path

import requests
from google.colab import userdata

def secret(name: str, default: str | None = None) -> str | None:
    try:
        return userdata.get(name) or default
    except Exception:
        return default

for name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"]:
    value = secret(name)
    if not value:
        raise RuntimeError(f"Mangler Colab Secret: {name}")
    os.environ[name] = value

github_token = secret("GITHUB_TOKEN") or getpass("GitHub token: ")
if not github_token:
    raise RuntimeError("GitHub token kreves for privat repo.")

for name in ["P12_OPENAI_MODEL", "P12_ANTHROPIC_MODEL", "P12_GEMINI_MODEL"]:
    value = secret(name)
    if value:
        os.environ[name] = value

os.environ["P12_RUN_ROOT"] = "/content/p12_synapse"
os.environ["P12_MOCK_MODE"] = "false"

In [ ]:
REPO = "nsolland/Tofoo-"
REF = "research/p12-synapse-understanding-tests"
RUNTIME = Path("/content/p12_runtime")
RUNTIME.mkdir(parents=True, exist_ok=True)

headers = {
    "Authorization": f"Bearer {github_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

def fetch_private_file(path: str) -> bytes:
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    response = requests.get(url, headers=headers, params={"ref": REF}, timeout=60)
    response.raise_for_status()
    payload = response.json()
    return base64.b64decode(payload["content"])

runner_bytes = fetch_private_file("experiments/P12_Synapse_Test/p12_runner.py")
tasks_bytes = fetch_private_file("experiments/P12_Synapse_Test/tasks.json")
corrections_bytes = fetch_private_file("experiments/P12_Synapse_Test/task_corrections.json")

runner_file = RUNTIME / "p12_runner.py"
tasks_file = RUNTIME / "tasks.json"
corrections_file = RUNTIME / "task_corrections.json"
runner_file.write_bytes(runner_bytes)
tasks_file.write_bytes(tasks_bytes)
corrections_file.write_bytes(corrections_bytes)

BASE_TASK_SHA = "e5eec7868ba9e8737c3f6a9200036819b97b9659df0a4e7c628c8d95ccffc05a"
EFFECTIVE_TASK_SHA = "0fa9a53a144cdc89875cc864c88d90d6c2cb535cff01dd86bad21be85311a6d9"
assert hashlib.sha256(tasks_bytes).hexdigest() == BASE_TASK_SHA

spec = importlib.util.spec_from_file_location("p12_runner", runner_file)
p12 = importlib.util.module_from_spec(spec)
sys.modules["p12_runner"] = p12
spec.loader.exec_module(p12)

TASKS = p12.load_tasks(tasks_file)
CORRECTIONS = json.loads(corrections_file.read_text(encoding="utf-8"))
assert CORRECTIONS["base_sha256"] == BASE_TASK_SHA
for correction in CORRECTIONS["corrections"]:
    task = next(t for t in TASKS if t["id"] == correction["task_id"])
    assert task[correction["field"]] == correction["old_value"]
    task[correction["field"]] = correction["new_value"]

assert p12.task_pack_sha256(TASKS) == EFFECTIVE_TASK_SHA
p12.CANONICAL_TASK_PACK_SHA256 = EFFECTIVE_TASK_SHA
p12.TASK_PACK_SHA256 = EFFECTIVE_TASK_SHA
p12.validate_config()

print("Oppgaver:", len(TASKS))
for model in p12.MODEL_POOL:
    print(f"{model.name}: {model.model}")
print("Dommere:", p12.JUDGE_NAMES)

## Smoke-test

In [ ]:
SMOKE_RUN = p12.run_experiment(TASKS, task_limit=2, label="smoke")
SMOKE_RUN

In [ ]:
smoke_report = p12.decision_report(SMOKE_RUN)
display(p12.budget_frame(SMOKE_RUN))
display(p12.judge_agreement(SMOKE_RUN))
smoke_report

## Full kjøring

In [ ]:
# Fjern kommentaren for full kjøring.
# FULL_RUN = p12.run_experiment(TASKS, task_limit=None, label="full")
# FULL_RUN

In [ ]:
# Etter full kjøring:
# full_report = p12.decision_report(FULL_RUN)
# p12.plot_summary(FULL_RUN)
# display(p12.budget_frame(FULL_RUN))
# display(p12.judge_agreement(FULL_RUN))
# full_report

## Gjenoppta avbrutt kjøring

In [ ]:
# FULL_RUN = p12.run_experiment(
#     TASKS,
#     task_limit=None,
#     existing_run_dir="/content/p12_synapse/20260802_120000_full",
# )

## Eksporter

In [ ]:
from google.colab import files

archive = p12.zip_run(SMOKE_RUN)
print(archive)
# files.download(str(archive))